In [0]:
pin_clean = spark.read.format('delta').table("038444ac863e_clean_pin")
geo_clean = spark.read.format('delta').table("038444ac863e_clean_geo")
user_clean = spark.read.format('delta').table("038444ac863e_clean_user")

In [0]:
pin_clean.createOrReplaceTempView("pin")
geo_clean.createOrReplaceTempView("geo")
user_clean.createOrReplaceTempView("user")

In [0]:
most_popular_cat_by_country = """
                        SELECT 
                        p.category,
                        g.country,
                        COUNT(p.category) as most_pop_cat
                        FROM pin p 
                        JOIN geo g
                        on p.ind == g.ind
                        GROUP BY g.country, p.category
                        ORDER BY most_pop_cat DESC
                        LIMIT 10;
                        """

display(spark.sql(most_popular_cat_by_country))

category,country,most_pop_cat
quotes,Algeria,10
tattoos,American Samoa,6
art,Albania,5
mens-fashion,Aruba,4
finance,Azerbaijan,4
christmas,Albania,4
tattoos,Andorra,4
education,Angola,4
christmas,Algeria,3
education,Afghanistan,3


In [0]:
most_popular_cat_by_year = """
                        SELECT 
                        p.category,
                        year(g.timestamp) as post_year,
                        COUNT(p.category) as yearly_post
                        FROM pin p  
                        JOIN geo g
                        on p.ind == g.ind
                        -- WHERE post_year BETWEEN 2018 AND 2022
                        GROUP BY p.category, post_year
                        HAVING post_year BETWEEN 2018 AND 2023
                        ORDER BY yearly_post DESC, post_year DESC
                        """

display(spark.sql(most_popular_cat_by_year))

category,post_year,yearly_post
art,2020,13
tattoos,2021,11
finance,2020,11
diy-and-crafts,2021,9
mens-fashion,2021,9
quotes,2021,9
education,2021,9
mens-fashion,2020,9
christmas,2020,9
tattoos,2020,9


In [0]:
top_country_by_followers = """
                        WITH max_followers_per_country AS (SELECT 
                        pin.poster_name,
                        MAX(pin.follower_count) as total_followers,
                        geo.country
                        FROM
                        pin
                        JOIN 
                        geo
                        ON
                        pin.ind == geo.ind
                        GROUP BY 
                        pin.poster_name,
                        geo.country
                        ORDER BY total_followers DESC)

                        SELECT country, total_followers
                        FROM max_followers_per_country
                        ORDER BY total_followers DESC
                        LIMIT 1
                        """

display(spark.sql(top_country_by_followers))

country,total_followers
American Samoa,8000000


In [0]:
most_popular_cat_by_age_group = """
    WITH age_agg AS (SELECT
    CASE
        WHEN u.age BETWEEN 18 and 24 THEN "18-24"
        WHEN u.age BETWEEN 25 and 35 THEN "25-35"
        WHEN u.age BETWEEN 36 and 50 THEN "36-50"
        WHEN u.age > 50 THEN "50+"
        END as age_group,
        p.category
    FROM user u
    JOIN pin p
    ON u.ind==p.ind),

most_pop_cat AS (SELECT COUNT(category) cat_count, age_group, category FROM age_agg GROUP BY age_group, category),
    ranking AS (select 
    age_group,
    category,
    cat_count,
    dense_rank() over (partition by age_group order by cat_count DESC) as rank
    from most_pop_cat)

    SELECT * FROM ranking
    WHERE rank = 1
    """

display(spark.sql(most_popular_cat_by_age_group))

age_group,category,cat_count,rank
18-24,tattoos,30,1
25-35,finance,12,1
36-50,finance,12,1
50+,mens-fashion,5,1


In [0]:
median_follower_count_by_age_group = """
    WITH user_age_followers AS (
    SELECT
        CASE
            WHEN u.age BETWEEN 18 and 24 THEN "18-24"
            WHEN u.age BETWEEN 25 and 35 THEN "25-35"
            WHEN u.age BETWEEN 36 and 50 THEN "36-50"
            WHEN u.age > 50 THEN "50+"
            END as age_group,
            p.follower_count
        FROM user u
        JOIN pin p
        ON u.ind==p.ind)

    SELECT 
        age_group,
        percentile_approx(follower_count, 0.5) AS median
        FROM user_age_followers
        GROUP BY age_group
        ORDER BY age_group;
    """

display(spark.sql(median_follower_count_by_age_group))

age_group,median
18-24,119000
25-35,25000
36-50,10000
50+,369


In [0]:
num_users_joined_by_year = """
   SELECT
        YEAR(date_joined) AS post_year,
        COUNT(*) AS number_users_joined
    FROM user
    WHERE 
        YEAR(date_joined) BETWEEN 2015 AND 2020
    GROUP BY
        YEAR(date_joined)
    ORDER BY
        post_year DESC;
    """

display(spark.sql(num_users_joined_by_year))

post_year,number_users_joined
2017,50
2016,168
2015,158


In [0]:
median_follower_joined_count_2015_2020 = """
   SELECT
        YEAR(user.date_joined) AS post_year,
        percentile_approx(pin.follower_count, 0.5) AS median_follower_count
    FROM user
    JOIN pin
        ON user.ind = pin.ind
    WHERE
        YEAR(user.date_joined) BETWEEN 2015 AND 2020
    GROUP BY
        YEAR(user.date_joined)
    """

display(spark.sql(median_follower_joined_count_2015_2020))

post_year,median_follower_count
2015,151000
2016,19000
2017,7000


In [0]:
median_follower_count_by_age_group_and_join_year = """
   SELECT
        CASE
            WHEN user.age >= 18 AND user.age <= 24 THEN '18-24'
            WHEN user.age >= 25 AND user.age <= 35 THEN '25-35'
            WHEN user.age >= 36 AND user.age <= 50 THEN '36-50'
            WHEN user.age > 50 THEN '50+'
        END AS age_group,
        YEAR(user.date_joined) AS post_year,
        percentile_approx(pin.follower_count, 0.5) AS median_follower_count
    FROM user
    JOIN pin 
        ON user.ind = pin.ind
    WHERE
        YEAR(user.date_joined) BETWEEN 2015 AND 2020
    GROUP BY
        age_group, post_year
    ORDER BY
        age_group, post_year DESC;
    """

display(spark.sql(median_follower_count_by_age_group_and_join_year))

age_group,post_year,median_follower_count
18-24,2017,7000
18-24,2016,41000
18-24,2015,267000
25-35,2017,2000
25-35,2016,24000
25-35,2015,42000
36-50,2017,7000
36-50,2016,13000
36-50,2015,10000
50+,2017,369
